## Inteligência de Escolha de Telefones no WhatsApp

Este Notebook implementa a solução solicitada no case, com foco em transformar logs históricos de disparo em uma regra de negócio acionável para priorização de telefones.

### Problema de negócio

Cada disparo tem custo e existe janela limitada para comunicação com o cidadão. Como o mesmo CPF pode ter múltiplos telefones oriundos de diferentes sistemas, o objetivo é escolher automaticamente os melhores números para maximizar entregas e reduzir custo por entrega.

### O que este Notebook entrega

1. Comparação de qualidade por sistema de origem com taxa e intervalo de confiança
2. Análise de decaimento da qualidade em função do tempo desde última atualização
3. Ranking de sistemas com suavização estatística para evitar conclusões instáveis
4. Algoritmo de score por telefone para escolher os dois melhores telefones por CPF
5. Proposta de experimento A B com hipótese, métricas e amostragem

### Definições operacionais

Sucesso de entrega é definido como `status_disparo` em `delivered` ou `read`. Nesta base, os status estão em minúsculo e `read` implica entrega.

Atualidade é aproximada por `registro_data_atualizacao` em `dim_telefone_mascarado.telefone_aparicoes`.

### Reprodutibilidade

Os Parquets devem estar em `data/raw` conforme instruções do README. Execute o Notebook a partir da raiz do repositório para garantir import do pacote `src`.

In [31]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt


# Garante que o pacote `src` seja importável mesmo se o Jupyter estiver com cwd em `notebooks`
_cwd = Path().resolve()
_repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.config import get_paths
from src.io import load_base_disparo, load_dim_telefone
from src.metrics import delivered_flag, aggregate_rate_with_ci
from src.scoring import SystemScoreParams, build_system_ranking, score_phones_for_cpf

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_float("full")

paths = get_paths(_repo_root)
paths

Paths(repo_root=WindowsPath('C:/Users/Nicole Pesoa/OneDrive/Desktop/desafio-tecnico/desafio-cientista-dados-pleno-campanhas'), data_raw=WindowsPath('C:/Users/Nicole Pesoa/OneDrive/Desktop/desafio-tecnico/desafio-cientista-dados-pleno-campanhas/data/raw'), data_processed=WindowsPath('C:/Users/Nicole Pesoa/OneDrive/Desktop/desafio-tecnico/desafio-cientista-dados-pleno-campanhas/data/processed'))

In [32]:
base_lf = load_base_disparo(paths.data_raw)
dim_lf = load_dim_telefone(paths.data_raw)

base_lf.collect_schema(), dim_lf.collect_schema()

(Schema([('id_conta', Int64),
         ('id_hsm', Int64),
         ('id_disparo', Int64),
         ('id_sessao', Int64),
         ('cpf', Int64),
         ('id_target', Int64),
         ('contato_telefone', Int64),
         ('categoria_hsm', String),
         ('ambiente', String),
         ('criacao_envio_datahora', Datetime(time_unit='us', time_zone=None)),
         ('envio_datahora', Datetime(time_unit='us', time_zone=None)),
         ('falha_datahora', Datetime(time_unit='us', time_zone=None)),
         ('descricao_falha', String),
         ('indicador_falha', Boolean),
         ('id_status_disparo', Int64),
         ('status_disparo', String)]),
 Schema([('telefone_ddi', String),
         ('telefone_ddd', Int64),
         ('telefone_numero', Int64),
         ('telefone_tipo', String),
         ('telefone_nacionalidade', String),
         ('telefone_qualidade', String),
         ('telefone_aparicoes',
          List(Struct({'id_sistema': String, 'cpf': Int64, 'proprietario_tipo': St

## Preparação do dataset analítico

A dimensão possui `telefone_aparicoes` como lista de structs. Um mesmo telefone pode aparecer em múltiplos sistemas, cada um com uma data de atualização própria. Para correlacionar sistema de origem com performance real de disparos, precisamos desnormalizar esse campo.

### Estratégia de modelagem

1. Explodir `telefone_aparicoes` para obter uma linha por telefone por sistema
2. Fazer join com a base de disparos usando o telefone como chave
3. Resolver o problema de multiplicidade do join. Um disparo pode casar com múltiplas linhas da dimensão, porque o telefone tem várias aparições

### Regra de seleção do sistema por disparo

Para cada `id_disparo`, escolhemos um único sistema representativo para atribuição de origem.

Critério

1. Preferir aparições com `registro_data_atualizacao` menor ou igual à data do envio, pois são temporalmente consistentes
2. Entre as consistentes, escolher a maior `registro_data_atualizacao`, pois representa o dado mais recente no momento do envio
3. Se nenhuma aparição for consistente, manter a mais recente disponível como fallback

Essa regra evita inflar a contagem do disparo em múltiplos sistemas ao mesmo tempo e torna as métricas por sistema comparáveis.

In [26]:
dim_aparicoes = (
    dim_lf
    .with_columns(telefone_mascarado=pl.col("telefone_numero"))
    .select(
        "telefone_numero",
        "telefone_ddi",
        "telefone_ddd",
        "telefone_tipo",
        "telefone_nacionalidade",
        "telefone_qualidade",
        "telefone_aparicoes",
        "telefone_aparicoes_quantidade",
        "telefone_proprietarios_quantidade",
        "telefone_sistemas_quantidade",
        "validacao_telefone",
    )
    .explode("telefone_aparicoes")
    .unnest("telefone_aparicoes")
    .rename(
        {
            "telefone_numero": "telefone_mascarado",
            "id_sistema": "id_sistema_mask",
            "registro_data_atualizacao": "registro_data_atualizacao",
            "cpf": "cpf_aparicao",
        }
    )
)

dim_aparicoes.collect_schema()

Schema([('telefone_mascarado', Int64),
        ('telefone_ddi', String),
        ('telefone_ddd', Int64),
        ('telefone_tipo', String),
        ('telefone_nacionalidade', String),
        ('telefone_qualidade', String),
        ('id_sistema_mask', String),
        ('cpf_aparicao', Int64),
        ('proprietario_tipo', String),
        ('registro_data_atualizacao', Date),
        ('telefone_aparicoes_quantidade', Int64),
        ('telefone_proprietarios_quantidade', Int64),
        ('telefone_sistemas_quantidade', Int64),
        ('validacao_telefone',
         Struct({'ddd_valido_br': Boolean, 'formato_valido': Boolean, 'padrao_suspeito': Boolean, 'padrao_invalido': Boolean}))])

In [4]:
base_enriched = (
    base_lf
    .with_columns(
        delivered=delivered_flag(pl.col("status_disparo")),
        envio_data=pl.col("envio_datahora").dt.date(),
    )
    .join(
        dim_aparicoes,
        left_on="contato_telefone",
        right_on="telefone_mascarado",
        how="left",
    )
    .with_columns(
        registro_data_atualizacao=pl.col("registro_data_atualizacao").cast(pl.Date),
        registro_valido_no_envio=(pl.col("registro_data_atualizacao") <= pl.col("envio_data")),
    )
)

base_enriched.select(
    "id_disparo",
    "contato_telefone",
    "status_disparo",
    "delivered",
    "envio_datahora",
    "envio_data",
    "id_sistema_mask",
    "registro_data_atualizacao",
    "registro_valido_no_envio",
).head(5).collect()

id_disparo,contato_telefone,status_disparo,delivered,envio_datahora,envio_data,id_sistema_mask,registro_data_atualizacao,registro_valido_no_envio
i64,i64,str,i8,datetime[μs],date,str,date,bool
-2317524427909960986,2824089259510570290,"""processing""",0,2025-10-07 13:15:17.204,2025-10-07,null,null,null
-6855906542267037066,-4599056651977889342,"""processing""",0,2025-10-25 10:56:45.028,2025-10-25,"""3094574413675758272""",2024-05-16,true
-6855906542267037066,-4599056651977889342,"""processing""",0,2025-10-25 10:56:45.028,2025-10-25,"""-133612832286195827""",null,null
-6855906542267037066,-4599056651977889342,"""processing""",0,2025-10-25 10:56:45.028,2025-10-25,"""-133612832286195827""",null,null
-6855906542267037066,-4599056651977889342,"""processing""",0,2025-10-25 10:56:45.028,2025-10-25,"""3094574413675758272""",2024-05-16,true


In [28]:
def select_system_for_event(lf: pl.LazyFrame) -> pl.LazyFrame:
    by_event = ["id_disparo"]

    candidates = (
        lf.with_columns(
            prefer=pl.when(pl.col("registro_valido_no_envio")).then(1).otherwise(0),
            recency=pl.col("registro_data_atualizacao"),
        )
        .sort(by_event + ["prefer", "recency"], descending=[False, True, True])
        .group_by(by_event)
        .first()
    )

    out = candidates.with_columns(
        age_days=(pl.col("envio_data") - pl.col("registro_data_atualizacao")).dt.total_days().cast(pl.Float64)
    )
    return out


events = select_system_for_event(base_enriched)

events.select(
    "id_disparo",
    "cpf",
    "contato_telefone",
    "id_sistema_mask",
    "registro_data_atualizacao",
    "envio_data",
    "age_days",
    "status_disparo",
    "delivered",
).head(5).collect()

id_disparo,cpf,contato_telefone,id_sistema_mask,registro_data_atualizacao,envio_data,age_days,status_disparo,delivered
i64,i64,i64,str,date,date,f64,str,i8
-9223351916581235936,5390755073247433751,4064757259083255873,"""1257277410380486863""",2025-11-01,2025-11-14,13,"""read""",1
-9223347553164632852,5727544527556597460,-5404120253297123491,"""-4704067261970591609""",2025-09-17,2025-12-12,86,"""read""",1
-9223331842594794498,2813023447562084872,2524702099694199688,"""3094574413675758272""",2025-10-02,2026-02-24,145,"""delivered""",1
-9223306595261143748,8905631595966597322,-19350203587627911,"""-4704067261970591609""",2025-06-18,2025-11-12,147,"""read""",1
-9223245723759412697,1818393646559723337,7546684854071138548,"""-4704067261970591609""",2025-10-31,2026-02-21,113,"""read""",1


In [29]:
# Consolida o dataset analítico selecionando as colunas necessárias
events_df = events.select(
    "id_disparo",
    "cpf",
    "contato_telefone",
    "id_sistema_mask",
    "registro_data_atualizacao",
    "envio_datahora",
    "envio_data",
    "age_days",
    "telefone_ddi",
    "telefone_ddd",
    "telefone_tipo",
    "telefone_qualidade",
    "validacao_telefone",
    "status_disparo",
    "delivered",
).collect()

# Salva o arquivo fisicamente na pasta data/processed
caminho_salvamento = paths.data_processed / "events_enriched.parquet"
events_df.write_parquet(caminho_salvamento)

print(f"Dataset analítico salvo com sucesso em: {caminho_salvamento}")

Dataset analítico salvo com sucesso em: C:\Users\Nicole Pesoa\OneDrive\Desktop\desafio-tecnico\desafio-cientista-dados-pleno-campanhas\data\processed\events_enriched.parquet
